# Занятие 27. Конструирование признаков

Алгоритм видит не квартиру, клиента или дату, а столбцы чисел. **Конструирование признаков** превращает исходные данные в представление, из которого модели легче извлечь закономерность.


## 1. Что делает признак хорошим

Хороший признак:

- известен в момент прогноза;
- одинаково вычисляется для train и новых объектов;
- связан с задачей по смыслу;
- не дублирует целевую переменную;
- устойчив к небольшим изменениям данных.

Сложный признак полезен только тогда, когда улучшает validation-качество, а не просто красиво звучит.


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'площадь': [35, 50, 80, 120],
    'комнаты': [1, 2, 3, 4],
    'цена': [7, 10, 16, 25],
    'район': ['центр', 'спальный', 'центр', 'пригород'],
    'дата': pd.to_datetime(['2025-01-10','2025-02-15','2025-06-01','2025-12-20'])
})
df


## 2. Числовые преобразования

Полезные операции:

- отношение: расходы на один день или площадь на одну комнату;
- разность: доход минус расходы;
- нормировка на размер: покупки на один день;
- `log1p(x)` для сильно скошенных неотрицательных величин;
- ограничение экстремальных значений после проверки их смысла.

Нельзя создавать признак из цели, если цель неизвестна в момент прогноза.


In [ ]:
df['площадь_на_комнату'] = df['площадь'] / df['комнаты']
incomes = np.array([20_000, 30_000, 50_000, 1_000_000])
log_incomes = np.log1p(incomes)
pd.DataFrame({'доход': incomes, 'log1p(доход)': log_incomes.round(2)})


## 3. Интервалы и ограничение выбросов

Иногда точное значение менее важно, чем диапазон: возраст 10–17, 18–24 и так далее. Интервалы создают ступенчатую зависимость и могут упростить линейной модели пороговые эффекты.

Экстремальные значения иногда ограничивают разумной границей, но только после проверки: настоящий редкий объект нельзя автоматически объявлять ошибкой.


In [ ]:
ages=pd.Series([12,17,18,24,25,67])
bins=pd.cut(ages,bins=[0,17,24,64,120],labels=['школьный','молодой','взрослый','старший'])
pd.DataFrame({'возраст':ages,'интервал':bins})


## 4. Нелинейность и взаимодействия

Это предварительный взгляд на линейную модель следующего занятия. Пока достаточно интуиции: простая формула складывает отдельные вклады признаков.

Новые признаки позволяют описать более сложные связи:

- $x^2$ — изгиб;
- $x_1x_2$ — влияние одного признака зависит от другого;
- интервалы — разные режимы.

Чем больше преобразований, тем выше риск переобучения.


In [ ]:
area = np.array([30, 60, 100])
balcony = np.array([0, 1, 1])
interaction = area * balcony
pd.DataFrame({'площадь': area, 'балкон': balcony, 'взаимодействие': interaction})


## 5. Как увидеть пользу взаимодействия

Взаимодействие нужно, когда влияние одного признака зависит от другого. На графике это проявляется разными наклонами или режимами для групп.

Произведения не добавляют автоматически: сначала формулируют смысловую гипотезу, затем проверяют её на validation.


In [ ]:
import matplotlib.pyplot as plt
area_line=np.array([30,50,70,90])
for balcony_value,color in [(0,'tab:blue'),(1,'tab:orange')]:
    effect=0.8*area_line+0.3*area_line*balcony_value
    plt.plot(area_line,effect,marker='o',color=color,label=f'балкон={balcony_value}')
plt.xlabel('Площадь'); plt.ylabel('Условный эффект')
plt.title('Одинаковые площади, но разные наклоны')
plt.grid(alpha=.3); plt.legend(); plt.show()


## 6. Даты и цикличность *

Из даты можно получить год, месяц, день недели, выходной, время с прошлого события.

Месяц 12 близок к месяцу 1, хотя числа далеко. Для циклических величин используют пару признаков:

$$\sin(2\pi m/12),\qquad \cos(2\pi m/12).$$

Эта тема отмечается `*`: для начала достаточно компонентов даты и осознания цикличности.


In [ ]:
df['месяц'] = df['дата'].dt.month
df['день_недели'] = df['дата'].dt.dayofweek
df['месяц_sin'] = np.sin(2*np.pi*df['месяц']/12)
df['месяц_cos'] = np.cos(2*np.pi*df['месяц']/12)
df[['дата','месяц','день_недели','месяц_sin','месяц_cos']].round(2)


## 7. Время с события

Часто полезнее не календарная дата, а прошедшее время: дней с регистрации, часов с последней покупки, возраст устройства.

Обе даты должны быть доступны в момент прогноза. Если «последнее событие» произошло после момента прогноза, возникает утечка.


In [ ]:
prediction_date=pd.Timestamp('2026-07-01')
last_event=pd.to_datetime(['2026-06-30','2026-06-01','2025-07-01'])
days_since=(prediction_date-last_event).days
pd.DataFrame({'последнее событие':last_event,'дней прошло':days_since})


## 8. Категориальные признаки

Названия районов нельзя бездумно заменить числами 1, 2, 3: модель увидит несуществующий порядок.

**One-hot encoding** создаёт отдельный столбец 0/1 для каждой категории. Для настоящего порядка — например, «низкий / средний / высокий» — допустимо порядковое кодирование, если порядок действительно задан смыслом.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

train_regions = np.array([['центр'], ['спальный'], ['пригород'], ['центр']])
new_regions = np.array([['центр'], ['новый район']])
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_encoded = encoder.fit_transform(train_regions)
new_encoded = encoder.transform(new_regions)
print('Столбцы:', encoder.get_feature_names_out())
print('Новые объекты:')
print(new_encoded)


## 9. Порядковые категории

У категорий «низкий / средний / высокий» есть настоящий порядок. Ordinal encoding может сохранить его числами 0, 1, 2.

Но модели используют числа по-разному: линейная модель тем самым предполагает одинаковый шаг между соседними уровнями, а дерево в основном использует пороги порядка. Если равный шаг не имеет смысла, сравнивают альтернативное кодирование по validation.

Для районов или цветов естественного порядка нет — там безопаснее one-hot encoding.


## 10. Редкие и неизвестные категории

Категория может встретиться в test, но отсутствовать в train. Кодировщик должен уметь обработать её без ошибки, например как «неизвестную».

Очень редкие категории часто объединяют в группу «другое». Правило объединения определяют по train и затем неизменно применяют к validation и test.


## 11. Пропуски

Пропуск не всегда означает ноль. Возможные стратегии:

- заполнить медианой числового признака;
- заполнить отдельной категорией;
- добавить индикатор `значение_пропущено`.

Иногда сам факт пропуска информативен. Параметры заполнения вычисляют только по train.


In [ ]:
values = pd.Series([10, np.nan, 14, 12, np.nan], name='значение')
missing_flag = values.isna().astype(int)
filled = values.fillna(values.median())  # в проекте медиану считаем только на train
pd.DataFrame({'исходное': values, 'заполненное': filled, 'был_пропуск': missing_flag})


## 12. Почему пропуски появились

Пропуск может быть случайной технической ошибкой, следствием процесса или сигналом о группе объектов. Например, поле дохода чаще не заполняют определённые клиенты.

Поэтому сравнивают распределения цели и других признаков для строк с пропуском и без него. Индикатор пропуска полезен, но тоже проверяется на validation.


## 13. Агрегаты по прошлым событиям

Для клиента полезны признаки «число прошлых покупок» и «средняя сумма прошлых заказов». Ключевое слово — **прошлых**.

Если в агрегат попали события после момента прогноза, возникла утечка.


## 14. Утечка через агрегаты и target encoding *

**Target encoding (*)** заменяет категорию средним ответом. Наивный расчёт сообщает учебному объекту часть его собственного ответа. Безопасный вариант требует специальных out-of-fold вычислений внутри train, поэтому пока достаточно запомнить предупреждение: самостоятельно считать среднее цели по всей таблице нельзя.

**Текстовые признаки — обзор.** Текст можно представить частотами слов или более сложными числовыми векторами. Словарь также обучают только на train; подробная работа с текстом относится к отдельной теме.


## 15. Масштабирование

StandardScaler приводит числовые признаки к сопоставимому масштабу. Это важно для:

- градиентных методов;
- моделей со штрафом за большие коэффициенты, которые мы изучим позже;
- методов, использующих расстояния.

Деревьям решений и случайному лесу масштабирование обычно не требуется. `fit` выполняют только на train.


## 16. Отбор и проверка признаков

Больше столбцов не всегда лучше. Лишние признаки увеличивают шум, время и риск утечки.

Проверяем новую идею честным экспериментом:

1. зафиксировать baseline-набор признаков;
2. добавить одну группу новых признаков;
3. обучить по тому же train;
4. сравнить по тому же validation;
5. проверить устойчивость на нескольких разбиениях.

Test для выбора признаков не используется.


## 17. Стоимость признака

Признак может улучшать качество, но быть слишком дорогим, медленным или недоступным в реальном времени. Учитывают стоимость получения, задержку, устойчивость и понятность.

Финальный набор — компромисс между качеством, риском утечки и эксплуатационной ценой, а не просто максимальное число столбцов.


## 18. Чек-лист конструирования признаков

1. Доступен ли признак в момент прогноза?
2. Нет ли в нём цели или будущего?
3. Одинаково ли он вычисляется для train и новых данных?
4. Как обрабатываются пропуски и неизвестные категории?
5. Нужен ли признаку масштаб?
6. Есть ли понятная гипотеза о пользе?
7. Улучшает ли он validation-качество устойчиво?

> **Главная мысль.** Признаки — это способ рассказать модели о задаче, не сообщая ей ответы из будущего.
